# Model Selection — Repair Cost

## Objective

Build and compare multiple regression models to predict vehicle repair cost.

Models evaluated:

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor

The best model will be selected based on:

- MAE
- RMSE
- R²

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [2]:
df = pd.read_csv("ml_ready.csv")

In [3]:
df.shape

(6583, 26)

In [4]:
X = df.drop(columns=["Repair_Cost","Turnaround_Time_Days","Is_Delayed"])

y = df["Repair_Cost"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [6]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5266, 23)
X_test : (1317, 23)
y_train: (5266,)
y_test : (1317,)


In [7]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "bool"]
).columns.tolist()

In [8]:
print("Numerical Features:", len(numerical_features))
print(numerical_features)

print("\nCategorical Features:", len(categorical_features))
print(categorical_features)

Numerical Features: 13
['Visit_Number', 'Vehicle_Age_at_Service', 'Battery_Age_at_Service', 'Battery_Health_at_Service', 'Expected_Part_ETA_Days', 'Active_Jobs_On_Arrival', 'Workshop_Utilization', 'Technician_Experience_Years', 'Repair_Complexity', 'Base_Labor_Hours', 'Technician_Efficiency', 'Effective_Labor_Hours', 'Expected_TAT_Days']

Categorical Features: 10
['Vehicle_Model', 'Battery_Replaced', 'Issue_Family', 'Exact_Issue', 'Parts_Required', 'Parts_Available', 'Part_Ordered', 'Day_Type', 'Warranty_Status', 'Warranty_Covered']


In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [10]:
linear_model = Pipeline(
    steps = [
        ("preprocessor",preprocessor),
        ("model",LinearRegression())
    ]
)

In [11]:
linear_model.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['Visit_Number',
                                                   'Vehicle_Age_at_Service',
                                                   'Battery_Age_at_Service',
                                                   'Battery_Health_at_Service',
                                                   'Expected_Part_ETA_Days',
                                                   'Active_Jobs_On_Arrival',
                                                   'Workshop_Utilization',
                                                   'Technician_Experience_Years',
                                                   'Repair_Complexity',
                                                   'Base_Labor_Hours',
                                                   'Technician_Efficiency',
                                                   'Effective_Labor_Hours',
                                                   'Expected_TAT_Days']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Vehicle_Model',
                                                   'Battery_Replaced',
                                                   'Issue_Family',
                                                   'Exact_Issue',
                                                   'Parts_Required',
                                                   'Parts_Available',
                                                   'Part_Ordered', 'Day_Type',
                                                   'Warranty_Status',
                                                   'Warranty_Covered'])])),
                ('model', LinearRegression())])

In [12]:
y_pred_linear = linear_model.predict(X_test)

In [13]:
mae_linear = mean_absolute_error(
    y_test,
    y_pred_linear
)

In [14]:
rmse_linear = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_linear
    )
)

In [15]:
r2_linear = r2_score(
    y_test,
    y_pred_linear
)

In [16]:
print(f"MAE  : {mae_linear:.2f}")
print(f"RMSE : {rmse_linear:.2f}")
print(f"R²   : {r2_linear:.3f}")

MAE  : 588.08
RMSE : 1204.19
R²   : 0.821


In [17]:
tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeRegressor(
            random_state=42
        ))
    ]
)

In [18]:
tree_model.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['Visit_Number',
                                                   'Vehicle_Age_at_Service',
                                                   'Battery_Age_at_Service',
                                                   'Battery_Health_at_Service',
                                                   'Expected_Part_ETA_Days',
                                                   'Active_Jobs_On_Arrival',
                                                   'Workshop_Utilization',
                                                   'Technician_Experience_Years',
                                                   'Repair_Complexity',
                                                   'Base_Labor_Hours',
                                                   'Technician_Efficiency',
                                                   'Effective_Labor_Hours',
                                                   'Expected_TAT_Days']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Vehicle_Model',
                                                   'Battery_Replaced',
                                                   'Issue_Family',
                                                   'Exact_Issue',
                                                   'Parts_Required',
                                                   'Parts_Available',
                                                   'Part_Ordered', 'Day_Type',
                                                   'Warranty_Status',
                                                   'Warranty_Covered'])])),
                ('model', DecisionTreeRegressor(random_state=42))])

In [19]:
y_pred_tree = tree_model.predict(X_test)

In [20]:
mae_tree = mean_absolute_error(
    y_test,
    y_pred_tree
)

rmse_tree = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_tree
    )
)

r2_tree = r2_score(
    y_test,
    y_pred_tree
)

print(f"MAE  : {mae_tree:.2f}")
print(f"RMSE : {rmse_tree:.2f}")
print(f"R²   : {r2_tree:.3f}")

MAE  : 386.84
RMSE : 883.90
R²   : 0.903


In [21]:
forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [22]:
forest_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['Visit_Number',
                                                   'Vehicle_Age_at_Service',
                                                   'Battery_Age_at_Service',
                                                   'Battery_Health_at_Service',
                                                   'Expected_Part_ETA_Days',
                                                   'Active_Jobs_On_Arrival',
                                                   'Workshop_Utilization',
                                                   'Technician_Experience_Years',
                                                   'Repair_Complexity',
                                                   'Base_Labor_Hours',
                                                   'Technician_Efficiency',
                                                   'Effective_Labor_Hours',
                                                   'Expected_TAT_Days']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Vehicle_Model',
                                                   'Battery_Replaced',
                                                   'Issue_Family',
                                                   'Exact_Issue',
                                                   'Parts_Required',
                                                   'Parts_Available',
                                                   'Part_Ordered', 'Day_Type',
                                                   'Warranty_Status',
                                                   'Warranty_Covered'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, n_jobs=-1,
                                       random_state=42))])

In [23]:
y_pred_forest = forest_model.predict(X_test)

In [24]:
mae_forest = mean_absolute_error(
    y_test,
    y_pred_forest
)

rmse_forest = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_forest
    )
)

r2_forest = r2_score(
    y_test,
    y_pred_forest
)

print(f"MAE  : {mae_forest:.2f}")
print(f"RMSE : {rmse_forest:.2f}")
print(f"R²   : {r2_forest:.3f}")

MAE  : 297.99
RMSE : 631.80
R²   : 0.951


In [25]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "MAE": [
        mae_linear,
        mae_tree,
        mae_forest
    ],
    "RMSE": [
        rmse_linear,
        rmse_tree,
        rmse_forest
    ],
    "R2": [
        r2_linear,
        r2_tree,
        r2_forest
    ]
})

results.sort_values("MAE")

,Model,MAE,RMSE,R2
2,Random Forest,297.988766,631.795071,0.950674
1,Decision Tree,386.835232,883.903850,0.903454
0,Linear Regression,588.078587,1204.192928,0.820809
